# gammanet contour-mask quadrant enrichment with matched bias

Deze notebook vervangt de oude quadrant-classificatie op basis van gemiddelde activatie per quadrant.

Nieuwe logica:
1. elke afbeelding wordt door VGG16-$\gamma$Net geforward;
2. voor C-afbeeldingen wordt optioneel C-channel bias toegepast; voor straight-afbeeldingen straight-channel bias;
3. uit `h1_exc` wordt een activatiemap gemaakt;
4. per afbeelding worden vier kandidaatmaskers getest: TL, TR, BL en BR;
5. per kandidaatmasker wordt contour enrichment berekend:

`mean activation inside contour mask / mean activation outside contour mask`

6. het masker/quadrant met de hoogste enrichment wordt gebruikt als voorspelling.


In [ ]:
# ============================================================
# 1. Imports, paths and settings
# ============================================================
import os, re, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

PROJECT_ROOT = Path("/home/yentl/pytorch_gammanet")
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoint_epoch_40.pt"

IMAGE_DIR = PROJECT_ROOT / "Images_quad_final"
MASK_DIR = PROJECT_ROOT / "contour_masks_quad_final"

OUTPUT_DIR = PROJECT_ROOT / "outputs_contour_mask_quadrant_enrichment_bias_025"
PLOT_DIR = OUTPUT_DIR / "plots"
CSV_DIR = OUTPUT_DIR / "csv"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SIZE = (320, 320)
ANALYSIS_LAYER = "h1_exc"
CONTOURS = ["C", "straight"]
QUADRANTS = ["TL", "TR", "BL", "BR"]

USE_ABS_ACTIVATION = True

# "matched_channels": C-afbeeldingen -> C_CHANNELS; straight-afbeeldingen -> STRAIGHT_CHANNELS.
# "all_channels": gemiddelde over alle h1-kanalen.
ACTIVATION_CHANNEL_MODE = "matched_channels"

RUN_BASELINE = True
RUN_MATCHED_BIAS = True
BIAS_STRENGTH = 0.25

transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
])

print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("ANALYSIS_LAYER:", ANALYSIS_LAYER)
print("ACTIVATION_CHANNEL_MODE:", ACTIVATION_CHANNEL_MODE)


In [ ]:
# ============================================================
# 2. Model, filename and mask helper functions
# ============================================================
def load_model():
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))

    from gammanet.models.vgg16_gammanet_v2 import VGG16GammaNetV2

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

    if "config" in checkpoint and "model" in checkpoint["config"]:
        model_config = checkpoint["config"]["model"]
    elif "model_config" in checkpoint:
        model_config = checkpoint["model_config"]
    else:
        raise KeyError("Could not find model config in checkpoint.")

    model = VGG16GammaNetV2(model_config)

    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", None))
    if state_dict is None:
        raise KeyError("Could not find model_state_dict or state_dict in checkpoint.")

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))

    model.to(DEVICE)
    model.eval()
    print("Model timesteps:", model.timesteps)
    return model


def normalize_contour_label(label):
    lower = str(label).strip().lower()
    if lower in ["c", "ccontour", "c_contour"]:
        return "C"
    if lower in ["straight", "line", "straightline", "straight_line"]:
        return "straight"
    return str(label)


def parse_image_filename(path):
    parts = path.stem.split("_")
    if len(parts) == 5:
        contour, quadrant, position, jitter, stim_id = parts
        contrast = "NA"
    elif len(parts) >= 6:
        contour, contrast, quadrant, position, jitter, stim_id = parts[:6]
    else:
        return None

    contour_type = normalize_contour_label(contour)
    quadrant = str(quadrant).upper()

    if contour_type not in CONTOURS or quadrant not in QUADRANTS:
        return None

    try:
        jitter_int = int(str(jitter).replace("J", ""))
    except Exception:
        jitter_int = np.nan
    try:
        position_int = int(position)
    except Exception:
        position_int = np.nan
    try:
        stim_id_int = int(stim_id)
    except Exception:
        stim_id_int = np.nan

    return {
        "filename": path.name,
        "path": str(path),
        "contour_type": contour_type,
        "contrast": contrast,
        "contour_quadrant": quadrant,
        "position": position_int,
        "jitter": jitter_int,
        "stimulus_id": stim_id_int,
    }


def candidate_mask_path(image_filename, candidate_quadrant):
    """Vervang alleen het quadrant-token in de bestandsnaam door TL/TR/BL/BR."""
    parts = Path(image_filename).stem.split("_")
    if len(parts) == 5:
        q_idx = 1
    elif len(parts) >= 6:
        q_idx = 2
    else:
        raise ValueError(f"Unexpected filename format: {image_filename}")
    parts[q_idx] = candidate_quadrant
    return MASK_DIR / ("_".join(parts) + Path(image_filename).suffix)


def load_image_tensor(path):
    pil_img = Image.open(path).convert("RGB")
    img_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)
    return pil_img, img_tensor


def load_binary_mask(mask_path, target_shape):
    mask_img = Image.open(mask_path).convert("L")
    mask = np.asarray(mask_img).astype(np.float32) > 0
    if mask.shape != target_shape:
        mask_t = torch.tensor(mask.astype(np.float32))[None, None]
        resized = F.interpolate(mask_t, size=target_shape, mode="nearest")
        mask = resized[0, 0].numpy() > 0.5
    return mask


def reset_and_forward(model, img_tensor):
    model.reset_hidden_states()
    with torch.no_grad():
        _ = model(img_tensor)


def get_state(model, layer_name):
    state = getattr(model, layer_name, None)
    if state is None:
        raise ValueError(f"{layer_name} is None. Run model first or check layer name.")
    return state


def normalize_for_plot(x, eps=1e-8):
    x = np.asarray(x)
    x = x - np.nanmin(x)
    return x / (np.nanmax(x) + eps)


In [ ]:
# ============================================================
# 3. Load C/straight channel classification
# ============================================================
CLASSIFICATION_DIR = (
    PROJECT_ROOT
    / "outputs_bias_contour_final"
    / "plots"
    / "05_channel_classification"
    / "channel_classification"
)

C_CHANNELS_CSV = CLASSIFICATION_DIR / "C_channels_h1.csv"
STRAIGHT_CHANNELS_CSV = CLASSIFICATION_DIR / "straight_channels_h1.csv"


def load_channel_list(csv_path):
    df = pd.read_csv(csv_path)
    if "channel" in df.columns:
        return df["channel"].astype(int).tolist()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 0:
        raise ValueError(f"No numeric channel column found in {csv_path}")
    return df[numeric_cols[0]].astype(int).tolist()


if not C_CHANNELS_CSV.exists() or not STRAIGHT_CHANNELS_CSV.exists():
    raise FileNotFoundError(
        "Channel classification files not found.\n"
        f"Expected C channels here: {C_CHANNELS_CSV}\n"
        f"Expected straight channels here: {STRAIGHT_CHANNELS_CSV}\n\n"
        "Run the channel-classification notebook first, or check CLASSIFICATION_DIR."
    )

C_CHANNELS = load_channel_list(C_CHANNELS_CSV)
STRAIGHT_CHANNELS = load_channel_list(STRAIGHT_CHANNELS_CSV)

print(f"Loaded {len(C_CHANNELS)} C channels from: {C_CHANNELS_CSV}")
print(f"Loaded {len(STRAIGHT_CHANNELS)} straight channels from: {STRAIGHT_CHANNELS_CSV}")
print("Bias strength:", BIAS_STRENGTH)


In [ ]:
# ============================================================
# 4. Activation map, enrichment and matched-bias functions
# ============================================================
def channel_map(state, contour_type=None, use_abs=True, channel_mode="matched_channels"):
    x = state.detach()
    if use_abs:
        x = x.abs()

    if channel_mode == "all_channels":
        return x.mean(dim=1)[0].cpu().numpy()

    if channel_mode == "matched_channels":
        contour_type = normalize_contour_label(contour_type)
        if contour_type == "C":
            channels = C_CHANNELS
        elif contour_type == "straight":
            channels = STRAIGHT_CHANNELS
        else:
            raise ValueError(f"Unknown contour_type: {contour_type}")
        idx = torch.tensor(channels, device=x.device, dtype=torch.long)
        return x[:, idx, :, :].mean(dim=1)[0].cpu().numpy()

    raise ValueError(f"Unknown channel_mode: {channel_mode}")


def resize_map_to_image(fmap, pil_img):
    fmap_t = torch.tensor(fmap, dtype=torch.float32)[None, None]
    resized = F.interpolate(
        fmap_t,
        size=pil_img.size[::-1],
        mode="bilinear",
        align_corners=False,
    )
    return resized[0, 0].numpy()


def contour_enrichment(fmap, mask, eps=1e-8):
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return {
            "activation_inside_mask": np.nan,
            "activation_outside_mask": np.nan,
            "contour_enrichment": np.nan,
            "mask_area_pixels": 0,
            "mask_area_fraction": 0.0,
        }
    mean_inside = float(np.nanmean(fmap[mask]))
    mean_outside = float(np.nanmean(fmap[~mask]))
    return {
        "activation_inside_mask": mean_inside,
        "activation_outside_mask": mean_outside,
        "contour_enrichment": float(mean_inside / (mean_outside + eps)),
        "mask_area_pixels": int(mask.sum()),
        "mask_area_fraction": float(mask.mean()),
    }


class TDH1Bias:
    def __init__(self, model, channels, strength=0.25, mode="add_mean_abs"):
        self.model = model
        self.channels = [int(c) for c in channels]
        self.strength = float(strength)
        self.mode = mode
        self.handle = None

    def _hook(self, module, inputs, output):
        if not isinstance(output, tuple):
            return output
        exc = output[0]
        if exc is None or len(self.channels) == 0:
            return output
        exc = exc.clone()
        idx = torch.tensor(self.channels, device=exc.device, dtype=torch.long)
        if self.mode == "add_mean_abs":
            scale = exc.detach().abs().mean(dim=(2, 3), keepdim=True) + 1e-8
            exc[:, idx, :, :] = exc[:, idx, :, :] + self.strength * scale[:, idx, :, :]
        elif self.mode == "multiply":
            exc[:, idx, :, :] = exc[:, idx, :, :] * (1.0 + self.strength)
        else:
            raise ValueError(f"Unknown bias mode: {self.mode}")
        return (exc,) + output[1:]

    def __enter__(self):
        self.handle = self.model.td_fgru_1.register_forward_hook(self._hook)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self.handle is not None:
            self.handle.remove()
        return False


def forward_condition(model, img_tensor, condition, contour_type):
    if condition == "baseline":
        return reset_and_forward(model, img_tensor)
    if condition == "matched_bias":
        contour_type = normalize_contour_label(contour_type)
        if contour_type == "C":
            with TDH1Bias(model, C_CHANNELS, strength=BIAS_STRENGTH):
                return reset_and_forward(model, img_tensor)
        if contour_type == "straight":
            with TDH1Bias(model, STRAIGHT_CHANNELS, strength=BIAS_STRENGTH):
                return reset_and_forward(model, img_tensor)
    raise ValueError(f"Unknown condition: {condition}")


In [ ]:
# ============================================================
# 5. Load model, image table and check masks
# ============================================================
model = load_model()

rows = []
for path in sorted(IMAGE_DIR.glob("*.png")):
    info = parse_image_filename(path)
    if info is not None:
        rows.append(info)

image_df = pd.DataFrame(rows)
if image_df.empty:
    raise RuntimeError(f"No valid C/straight images found in {IMAGE_DIR}.")

image_df = image_df.sort_values(["contour_type", "contour_quadrant", "filename"]).reset_index(drop=True)

missing_masks = []
for _, row in image_df.iterrows():
    for q in QUADRANTS:
        mp = candidate_mask_path(row["filename"], q)
        if not mp.exists():
            missing_masks.append({"filename": row["filename"], "candidate_quadrant": q, "expected_mask": str(mp)})

missing_masks_df = pd.DataFrame(missing_masks)

print("Number of images:", len(image_df))
print(image_df.groupby(["contour_type", "contour_quadrant"]).size())
print("Missing candidate masks:", len(missing_masks_df))

if not missing_masks_df.empty:
    display(missing_masks_df.head(20))
    raise FileNotFoundError("Some candidate masks are missing. See missing_masks_df above.")

display(image_df.head())


In [ ]:
# ============================================================
# 6. Main analysis: classify quadrant by highest contour enrichment
# ============================================================
conditions = []
if RUN_BASELINE:
    conditions.append("baseline")
if RUN_MATCHED_BIAS:
    conditions.append("matched_bias")

records = []
activation_maps = {}

for condition in conditions:
    print("Running condition:", condition)
    for _, row in tqdm(image_df.iterrows(), total=len(image_df), desc=condition):
        pil_img, img_tensor = load_image_tensor(row["path"])

        forward_condition(model, img_tensor, condition, row["contour_type"])
        state = get_state(model, ANALYSIS_LAYER)
        fmap = channel_map(
            state,
            contour_type=row["contour_type"],
            use_abs=USE_ABS_ACTIVATION,
            channel_mode=ACTIVATION_CHANNEL_MODE,
        )
        fmap_resized = resize_map_to_image(fmap, pil_img)
        activation_maps[(condition, row["filename"])] = fmap_resized

        row_record = {
            "filename": row["filename"],
            "path": row["path"],
            "contour_type": row["contour_type"],
            "true_contour_quadrant": row["contour_quadrant"],
            "contrast": row["contrast"],
            "position": row["position"],
            "jitter": row["jitter"],
            "stimulus_id": row["stimulus_id"],
            "condition": condition,
            "bias_strength": BIAS_STRENGTH if condition == "matched_bias" else 0.0,
            "activation_layer": ANALYSIS_LAYER,
            "activation_channel_mode": ACTIVATION_CHANNEL_MODE,
            "use_abs_activation": USE_ABS_ACTIVATION,
        }

        enrichments = {}
        for q in QUADRANTS:
            mask_path = candidate_mask_path(row["filename"], q)
            mask = load_binary_mask(mask_path, target_shape=fmap_resized.shape)
            scores = contour_enrichment(fmap_resized, mask)
            enrichments[q] = scores["contour_enrichment"]
            row_record[f"mask_{q}_path"] = str(mask_path)
            row_record[f"enrichment_{q}"] = scores["contour_enrichment"]
            row_record[f"inside_activation_{q}"] = scores["activation_inside_mask"]
            row_record[f"outside_activation_{q}"] = scores["activation_outside_mask"]
            row_record[f"mask_area_fraction_{q}"] = scores["mask_area_fraction"]

        predicted_q = max(enrichments, key=enrichments.get)
        row_record["predicted_quadrant"] = predicted_q
        row_record["highest_contour_enrichment"] = enrichments[predicted_q]
        row_record["matches_true_quadrant"] = "yes" if predicted_q == row["contour_quadrant"] else "no"
        records.append(row_record)

results_df = pd.DataFrame(records)
results_df = results_df.sort_values(["condition", "contour_type", "true_contour_quadrant", "filename"]).reset_index(drop=True)

results_path = CSV_DIR / "h1_contour_mask_enrichment_quadrant_results.csv"
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
display(results_df.head())


In [ ]:
# ============================================================
# 7. Summary tables
# ============================================================
summary_df = (
    results_df
    .groupby(["condition", "contour_type", "true_contour_quadrant", "activation_channel_mode"])
    .agg(
        n_images=("filename", "count"),
        n_match=("matches_true_quadrant", lambda x: (x == "yes").sum()),
        proportion_match=("matches_true_quadrant", lambda x: (x == "yes").mean()),
        mean_highest_enrichment=("highest_contour_enrichment", "mean"),
        mean_enrichment_TL=("enrichment_TL", "mean"),
        mean_enrichment_TR=("enrichment_TR", "mean"),
        mean_enrichment_BL=("enrichment_BL", "mean"),
        mean_enrichment_BR=("enrichment_BR", "mean"),
    )
    .reset_index()
)
summary_path = CSV_DIR / "h1_contour_mask_enrichment_quadrant_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(summary_df)

overall_summary_df = (
    results_df
    .groupby(["condition", "contour_type", "activation_channel_mode"])
    .agg(
        n_images=("filename", "count"),
        n_match=("matches_true_quadrant", lambda x: (x == "yes").sum()),
        proportion_match=("matches_true_quadrant", lambda x: (x == "yes").mean()),
        mean_highest_enrichment=("highest_contour_enrichment", "mean"),
    )
    .reset_index()
)
overall_summary_path = CSV_DIR / "h1_contour_mask_enrichment_overall_summary.csv"
overall_summary_df.to_csv(overall_summary_path, index=False)
print("Saved:", overall_summary_path)
display(overall_summary_df)


In [ ]:
# ============================================================
# 8. Confusion matrices: true quadrant x predicted quadrant
# ============================================================
for condition in results_df["condition"].unique():
    for contour in results_df["contour_type"].unique():
        subset = results_df[(results_df["condition"] == condition) & (results_df["contour_type"] == contour)]
        conf = pd.crosstab(
            subset["true_contour_quadrant"],
            subset["predicted_quadrant"],
            rownames=["true"],
            colnames=["predicted"],
            dropna=False,
        ).reindex(index=QUADRANTS, columns=QUADRANTS, fill_value=0)
        save_path = CSV_DIR / f"confusion_{condition}_{contour}.csv"
        conf.to_csv(save_path)
        print(f"\n{condition} - {contour}")
        display(conf)
        print("Saved:", save_path)


In [ ]:
# ============================================================
# 9. Plot: proportion correct per contour and condition
# ============================================================
plot_df = overall_summary_df.copy()

plt.figure(figsize=(7, 4))
for contour in CONTOURS:
    subset = plot_df[plot_df["contour_type"] == contour]
    if subset.empty:
        continue
    plt.plot(subset["condition"], subset["proportion_match"], marker="o", linewidth=2, label=contour)

plt.axhline(0.25, linestyle="--", linewidth=1, label="chance")
plt.ylim(0, 1)
plt.xlabel("Condition")
plt.ylabel("Proportion correct")
plt.title("Quadrant classification by highest contour-mask enrichment")
plt.legend()
plt.tight_layout()
plot_path = PLOT_DIR / "classification_accuracy_by_contour_and_condition.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)


In [ ]:
# ============================================================
# 10. Optional visual check for one example image
# ============================================================
example_index = 0
example_condition = conditions[-1]

example = image_df.iloc[example_index]
example_fmap = activation_maps[(example_condition, example["filename"])]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(normalize_for_plot(example_fmap), cmap="magma")
axes[0].set_title(f"Activation\n{example_condition}\n{example['filename']}")
axes[0].axis("off")

for ax, q in zip(axes[1:], QUADRANTS):
    mask_path = candidate_mask_path(example["filename"], q)
    mask = load_binary_mask(mask_path, target_shape=example_fmap.shape)
    score = contour_enrichment(example_fmap, mask)["contour_enrichment"]
    ax.imshow(normalize_for_plot(example_fmap), cmap="magma")
    ax.contour(mask.astype(float), levels=[0.5], linewidths=1)
    ax.set_title(f"Mask {q}\nenrichment={score:.3f}")
    ax.axis("off")

plt.tight_layout()
visual_check_path = PLOT_DIR / f"visual_check_{example_condition}_{example['filename'].replace('.png', '')}.png"
plt.savefig(visual_check_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", visual_check_path)


## Belangrijkste output

Na het runnen staan de belangrijkste bestanden hier:

- `outputs_contour_mask_quadrant_enrichment_bias_025/csv/h1_contour_mask_enrichment_quadrant_results.csv`  
  Per afbeelding: enrichment voor TL/TR/BL/BR, voorspeld quadrant, en match `yes/no`.

- `outputs_contour_mask_quadrant_enrichment_bias_025/csv/h1_contour_mask_enrichment_quadrant_summary.csv`  
  Samenvatting per conditie, contourtype en echt quadrant.

- `outputs_contour_mask_quadrant_enrichment_bias_025/csv/h1_contour_mask_enrichment_overall_summary.csv`  
  Globale samenvatting per conditie en contourtype.

- `outputs_contour_mask_quadrant_enrichment_bias_025/csv/confusion_<condition>_<contour>.csv`  
  Confusion matrix per conditie en contourtype.
